In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/vi-sub-correction')
CKPT_ROOT  = DRIVE_ROOT / 'checkpoints'
DATA_DRIVE = DRIVE_ROOT / 'data'
for d in (CKPT_ROOT, DATA_DRIVE):
    d.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/vi-sub-corretion')
DATA_DIR = REPO_DIR / 'data' / 'synthetic'
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Da ket noi Drive.')
print('Checkpoint ben vung :', CKPT_ROOT)
print('Du lieu tam (/content):', DATA_DIR)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

if not REPO_DIR.exists():
    !git clone https://github.com/lhhoang5425/vi-sub-corretion.git /content/vi-sub-corretion

%cd /content/vi-sub-corretion
!git pull
!pip install -r requirements.txt -q
!pip install -q openpyxl

print('--- 3 commit gan nhat ma Colab dang chay ---')
!git log --oneline -3

In [ ]:
import json
import re
from datasets import load_dataset
from src.data_processing.corpus_builder import build_noisy_corpus, format_stats, write_jsonl

# 100k câu SẠCH (nạp dư vì nguồn trùng lặp ~21%), sinh 3 hồ sơ nhiễu để so sánh.
NUM_TRAIN_CLEAN = 100_000
NUM_FETCH = 220_000
SEED = 42
VARIANTS = ['legacy', 'asr_hard', 'whisper_like']
VAL_RATIO = 0.1

raw = load_dataset('ShynBui/Vietnamese_spelling_error', split=f'train[:{NUM_FETCH}]')
clean = [(r['text'] or '').strip() for r in raw]

# Khử trùng lặp rồi chia theo hash nội dung -> không rò rỉ val<->train (lỗi của Lần 4)
corpus_by_profile = {
    p: build_noisy_corpus(clean, profile=p, val_ratio=VAL_RATIO, seed=SEED, max_clean=NUM_TRAIN_CLEAN)
    for p in VARIANTS
}
print(format_stats(corpus_by_profile[VARIANTS[0]]['stats']))

TONE = re.compile(r'[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]', re.I)
gold = [json.loads(l) for l in open('data/gold_test/gd1_basic/sample_test.jsonl', encoding='utf-8') if l.strip()]
gold_lost = sum(1 for g in gold if not TONE.search(g['noisy'])) / len(gold) * 100
print(f'\nĐề thi gốc yêu cầu khôi phục dấu thanh cho {gold_lost:.0f}% câu\n')

print(f'{"HỒ SƠ":<14}{"train":>9}{"val":>8}{"% mất trắng dấu":>18}')
print('-' * 50)
for prof in VARIANTS:
    c = corpus_by_profile[prof]
    n_tr = write_jsonl(c['train'], DATA_DIR / f'train_{prof}.jsonl')
    n_va = write_jsonl(c['validation'], DATA_DIR / f'val_{prof}.jsonl')
    rows = c['validation'][:500]
    lost = sum(1 for r in rows if not TONE.search(r['noisy'])) / len(rows) * 100
    print(f'{prof:<14}{n_tr:>9,}{n_va:>8,}{lost:>17.1f}%')

TRAIN_HARD, VAL_HARD = str(DATA_DIR / 'train_asr_hard.jsonl'), str(DATA_DIR / 'val_asr_hard.jsonl')
TRAIN_WHISPER, VAL_WHISPER = str(DATA_DIR / 'train_whisper_like.jsonl'), str(DATA_DIR / 'val_whisper_like.jsonl')
TRAIN_LEG, VAL_LEG = str(DATA_DIR / 'train_legacy.jsonl'), str(DATA_DIR / 'val_legacy.jsonl')

!cp {DATA_DIR / 'train_asr_hard.jsonl'} {DATA_DRIVE / 'train_asr_hard.jsonl'}
!cp {DATA_DIR / 'train_whisper_like.jsonl'} {DATA_DRIVE / 'train_whisper_like.jsonl'}
print('\nĐã copy dữ liệu train sang Drive.')


In [ ]:
import re
import zipfile

# Giải nén checkpoint Lần 4 làm MỐC SO SÁNH.
# Chỉ lấy weights + tokenizer ở thư mục gốc, bỏ checkpoint-*/ (optimizer ~2.5GB/epoch).
REF = CKPT_ROOT / 'ref_lan4_legacy'

# Tìm tệp zip của bản Lần 4: chỉ cần trong tên có "bmd1905", ưu tiên tên có số 4.
# Đổi tên tệp thế nào cũng được, cell này tự dò.
zips = [p for p in CKPT_ROOT.glob('*.zip') if 'bmd1905' in p.name.lower()]
if not zips:
    raise FileNotFoundError(
        f'Không thấy tệp .zip nào chứa "bmd1905" trong {CKPT_ROOT}\n'
        f'  Đặt tên thế nào cũng được, miễn trong tên có "bmd1905".'
    )
lan4 = [p for p in zips if re.search(r'(lan|train)[^0-9]*4', p.name, re.I)]
REF_ZIP = (lan4 or zips)[0]
print(f'Mốc so sánh: {REF_ZIP.name}')
if len(zips) > 1:
    print(f'  (có {len(zips)} tệp cùng khớp: {[p.name for p in zips]})')

if (REF / 'config.json').exists():
    print(f'Đã giải nén sẵn: {REF}')
else:
    REF.mkdir(parents=True, exist_ok=True)
    print(f'Đang giải nén ({REF_ZIP.stat().st_size / 1024**3:.2f} GB) ...')
    with zipfile.ZipFile(REF_ZIP) as z:
        names = z.namelist()
        # Tự dò thư mục gốc từ vị trí config.json thay vì khớp cứng đường dẫn bên trong zip
        anchor = next((n for n in names
                       if n.endswith('config.json') and '/checkpoint-' not in n), None)
        if anchor is None:
            raise RuntimeError(
                f'{REF_ZIP.name} không chứa config.json — không phải checkpoint đã lưu.\n'
                f'  Vài mục đầu trong zip: {names[:5]}'
            )
        root = anchor[: -len('config.json')]
        picked = [n for n in names
                  if not n.endswith('/') and n.startswith(root) and '/checkpoint-' not in n]
        for n in picked:
            dest = REF / n[len(root):]
            dest.parent.mkdir(parents=True, exist_ok=True)
            with z.open(n) as src, open(dest, 'wb') as out:
                out.write(src.read())
    total = sum(f.stat().st_size for f in REF.rglob('*') if f.is_file()) / 1024**3
    print(f'Xong: {total:.2f} GB — {len(picked)} tệp')


In [ ]:
import shutil
import subprocess
import sys as _sys

# Corpus văn học để đo OVER-CORRECTION (không nằm trong git vì bản quyền).
CORPUS_DRIVE = DRIVE_ROOT / 'corpus'
CORPUS_LOCAL = REPO_DIR / 'corpus'

if CORPUS_DRIVE.exists() and any(CORPUS_DRIVE.glob('*.txt')):
    shutil.copytree(CORPUS_DRIVE, CORPUS_LOCAL, dirs_exist_ok=True)
    print(f'Đã đồng bộ {len(list(CORPUS_LOCAL.glob("*.txt")))} tệp .txt từ Drive')
    subprocess.run([_sys.executable, '-m', 'src.data_processing.build_corpus_test',
                    '--out_dir', 'data/gold_test/control'], check=True)
    subprocess.run([_sys.executable, '-m', 'src.evaluation.run_benchmark',
                    '--test_file', 'data/gold_test/control/corpus_identity.jsonl',
                    '--model_name_or_path', str(REF),
                    '--output_json', str(REF / 'metrics_corpus_identity.json'),
                    '--batch_size', '16'], check=True)
else:
    print('Chưa có corpus trên Drive — bỏ qua phần này.')


In [ ]:
import json
from src.models.inference import Seq2SeqCorrector
from src.evaluation.metrics import evaluate_predictions

# Grid-search tham số giải mã trên VAL — không train lại, không đụng gold test.
# Train cho gd2 thì đổi: VAL_HARD -> VAL_WHISPER
GRID_VAL = VAL_HARD
dev = [json.loads(l) for l in open(GRID_VAL, encoding='utf-8')][:500]
srcs = [r['noisy'] for r in dev]
refs = [r['clean'] for r in dev]
print(f'Grid-search trên {len(dev)} câu VAL\n')

GRID = [
    {'num_beams': 5, 'repetition_penalty': 1.20, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3},
    {'num_beams': 5, 'repetition_penalty': 1.00, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3},
    {'num_beams': 5, 'repetition_penalty': 1.05, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0},
    {'num_beams': 8, 'repetition_penalty': 1.00, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3},
    {'num_beams': 4, 'repetition_penalty': 1.00, 'length_penalty': 1.2, 'no_repeat_ngram_size': 3},
]

decoder = Seq2SeqCorrector(model_name_or_path=str(REF), use_fallback=False)

hdr = f'{"beams":>6}{"rep_pen":>9}{"len_pen":>9}{"ngram":>7}{"EM%":>8}{"BLEU":>8}{"WER%":>8}{"CER%":>8}{"Ed-F1%":>9}'
print(hdr)
print('-' * len(hdr))
scored = []
for cfg in GRID:
    decoder.set_decoding(**cfg)
    preds = decoder.predict_batch(srcs, batch_size=32)
    m = evaluate_predictions(srcs, preds, refs)
    scored.append((m['exact_match'], m['edit_f1'], cfg, m))
    print(f'{cfg["num_beams"]:>6}{cfg["repetition_penalty"]:>9.2f}{cfg["length_penalty"]:>9.1f}'
          f'{cfg["no_repeat_ngram_size"]:>7}{m["exact_match"]:>8.2f}{m["bleu"]:>8.2f}'
          f'{m["wer"]:>8.2f}{m["cer"]:>8.2f}{m["edit_f1"]:>9.2f}'
          + ('   <- đang dùng' if cfg is GRID[0] else ''))

scored.sort(key=lambda t: (t[0], t[1]), reverse=True)
BEST_DECODING = scored[0][2]
BEAMS = BEST_DECODING['num_beams']
REP   = BEST_DECODING['repetition_penalty']
LENP  = BEST_DECODING['length_penalty']
NGRAM = BEST_DECODING['no_repeat_ngram_size']
print(f'\nCHỐT: beams={BEAMS} rep_pen={REP} len_pen={LENP} ngram={NGRAM}')


In [ ]:
import subprocess
import sys as _sys
from pathlib import Path

# Bộ kết quả của MỐC Lần 4 trên mọi tập test, dùng đúng cách giải mã đã chốt ở cell 5.
TEST_SUITES = [
    ('sample25',    'data/gold_test/gd1_basic/sample_test.jsonl'),
    ('unaccent',    'data/gold_test/gd1_basic/basic_test_unaccent_all.jsonl'),
    ('asrhard',     'data/gold_test/gd1_basic/basic_test_asr_hard.jsonl'),
    ('multisource', 'data/gold_test/gd1_basic/basic_test_multisource.jsonl'),
    ('identity',    'data/gold_test/control/corpus_identity.jsonl'),
]

for tag, path in TEST_SUITES:
    if not Path(path).exists():
        print(f'[BỎ QUA] chưa có {path}')
        continue
    print(f'\n{"=" * 70}\n  {tag}\n{"=" * 70}')
    subprocess.run([_sys.executable, '-m', 'src.evaluation.run_benchmark',
                    '--test_file', path,
                    '--model_name_or_path', str(REF),
                    '--output_report', str(REF / f'benchmark_{tag}.md'),
                    '--output_json',   str(REF / f'metrics_{tag}.json'),
                    '--output_log',    str(REF / f'benchmark_{tag}_log.txt'),
                    '--batch_size', '16',
                    '--num_beams', str(BEAMS),
                    '--repetition_penalty', str(REP),
                    '--length_penalty', str(LENP),
                    '--no_repeat_ngram_size', str(NGRAM)], check=True)
print('\nXong bộ kết quả mốc Lần 4.')


In [ ]:
import json
from datetime import datetime

# Manifest của mốc Lần 4: chép từ log Colab vì Lần 4 chạy trước khi có cơ chế manifest.
# Cần để compare_runs đưa được mốc cũ vào bảng so sánh.
(REF / 'manifest.json').write_text(json.dumps({
    'run_name': REF.name,
    'created_at': datetime.now().astimezone().isoformat(timespec='seconds'),
    'ghi_chu': 'ĐIỀN THỦ CÔNG từ log Lần 4, không do script tự sinh',
    'base_model': 'bmd1905/vietnamese-correction-v2',
    'device': 'cuda',
    'gpu': 'Tesla T4',
    'hyperparams': {'num_train_epochs': 3, 'per_device_train_batch_size': 16,
                    'learning_rate': 3e-05, 'max_length': 128, 'label_smoothing': 0.0,
                    'padding': 'static', 'fp16': True},
    'dataset': {'train_size': 90000, 'val_size': 10000},
    'results': {'train_loss_final': 0.03736,
                'eval_loss_per_epoch': [0.02764, 0.01632, 0.01307],
                'best_eval_loss': 0.01307,
                'train_runtime_sec': 7941.0,
                'total_steps': 16875},
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Đã ghi {REF / "manifest.json"}')


In [ ]:
from src.models.train_seq2seq import resolve_output_dir

# ============== CHỌN BÀI TOÁN Ở ĐÂY — CHỈ ĐỔI DÒNG NÀY ==============
#   'asr_hard'     -> gd1: khôi phục dấu từ văn bản MẤT dấu
#   'whisper_like' -> gd2: sửa lỗi ASR thật (SAI dấu, nhầm âm)
#   'legacy'       -> như Lần 4, để đối chứng
VARIANT = 'asr_hard'
# ====================================================================

DATA_OF = {
    'asr_hard':     (TRAIN_HARD,    VAL_HARD,    'asr_hard'),
    'whisper_like': (TRAIN_WHISPER, VAL_WHISPER, 'whisper'),
    'legacy':       (TRAIN_LEG,     VAL_LEG,     'legacy'),
}
TRAIN_F, VAL_F, TAG = DATA_OF[VARIANT]
LABEL_SMOOTHING = 0.1
RUN_TAG = f'lan5_{TAG}_ls{int(LABEL_SMOOTHING * 10)}'
RUN_DIR_BASE = str(CKPT_ROOT / 'bmd1905-sub-corrector')

print(f'Bài toán    : {VARIANT}')
print(f'Dữ liệu     : {TRAIN_F}')

# Chống đè: xem các đợt đã có, rồi chốt tên thư mục đích TRƯỚC khi train.
!python -m src.models.train_seq2seq --list_runs --output_dir "{RUN_DIR_BASE}"

DEST = resolve_output_dir(RUN_DIR_BASE, RUN_TAG, overwrite=False)
print(f'\nĐỢT NÀY SẼ LƯU VÀO: {DEST}\n')

!python -m src.models.train_seq2seq \
    --model_name 'bmd1905/vietnamese-correction-v2' \
    --train_file '{TRAIN_F}' \
    --val_file   '{VAL_F}' \
    --output_dir '{DEST}' \
    --epochs 3 --batch_size 16 --lr 3e-5 \
    --label_smoothing {LABEL_SMOOTHING}


In [ ]:
import subprocess
import sys as _sys
from pathlib import Path

# Đo đợt mới trên MỌI tập test -> biết ngay mô hình giỏi bài toán nào.
# Tập 'multisource' có bảng riêng theo NGUỒN (shynbui / wikipedia).
TEST_SUITES = [
    ('sample25',    'data/gold_test/gd1_basic/sample_test.jsonl'),
    ('unaccent',    'data/gold_test/gd1_basic/basic_test_unaccent_all.jsonl'),
    ('asrhard',     'data/gold_test/gd1_basic/basic_test_asr_hard.jsonl'),
    ('multisource', 'data/gold_test/gd1_basic/basic_test_multisource.jsonl'),
    ('kichban',     'data/gold_test/gd2_real_asr/kichban_test.jsonl'),
    ('identity',    'data/gold_test/control/corpus_identity.jsonl'),
]

print(f'Mô hình: {DEST} | Bài toán: {VARIANT}')
for tag, path in TEST_SUITES:
    if not Path(path).exists():
        print(f'[BỎ QUA] chưa có {path}')
        continue
    print(f'\n{"=" * 70}\n  {tag}\n{"=" * 70}')
    subprocess.run([_sys.executable, '-m', 'src.evaluation.run_benchmark',
                    '--test_file', path,
                    '--model_name_or_path', str(DEST),
                    '--output_report', str(DEST / f'benchmark_{tag}.md'),
                    '--output_json',   str(DEST / f'metrics_{tag}.json'),
                    '--output_log',    str(DEST / f'benchmark_{tag}_log.txt'),
                    '--batch_size', '16',
                    '--num_beams', str(BEAMS),
                    '--repetition_penalty', str(REP),
                    '--length_penalty', str(LENP),
                    '--no_repeat_ngram_size', str(NGRAM)], check=True)
print('\nXong.')


In [ ]:
# ĐẶT CẠNH NHAU: mốc Lần 4 vs đợt mới — cùng chỉ số, cùng cách giải mã.
!python -m src.evaluation.compare_runs \
    --runs_root "{CKPT_ROOT}" \
    --output_md "{CKPT_ROOT / 'so_sanh_dot_train.md'}" \
    --sort_by exact_match

from google.colab import files
files.download(str(CKPT_ROOT / 'so_sanh_dot_train.md'))


In [ ]:
# TÙY CHỌN: tách riêng đóng góp của TỪNG thay đổi (mỗi đợt ~1-2 giờ).
# Bỏ qua nếu Colab sắp hết phiên.
from src.models.train_seq2seq import resolve_output_dir

# (B) asr_hard KHÔNG label smoothing -> cô lập tác động của LS
DEST_B = resolve_output_dir(str(CKPT_ROOT / 'bmd1905-sub-corrector'), 'lan5_asr_hard_ls00', overwrite=False)
!python -m src.models.train_seq2seq \
    --model_name 'bmd1905/vietnamese-correction-v2' \
    --train_file '{TRAIN_HARD}' --val_file '{VAL_HARD}' \
    --output_dir '{DEST_B}' \
    --epochs 3 --batch_size 16 --lr 3e-5 --label_smoothing 0.0

# (C) nhiễu LEGACY (như Lần 4) nhưng CÓ pad động + LS -> cô lập tác động của NHIỄU
DEST_C = resolve_output_dir(str(CKPT_ROOT / 'bmd1905-sub-corrector'), 'lan5_legacy_ls01', overwrite=False)
!python -m src.models.train_seq2seq \
    --model_name 'bmd1905/vietnamese-correction-v2' \
    --train_file '{TRAIN_LEG}' --val_file '{VAL_LEG}' \
    --output_dir '{DEST_C}' \
    --epochs 3 --batch_size 16 --lr 3e-5 --label_smoothing 0.1


In [ ]:
# Đo 2 biến thể phụ rồi mở lại bảng so sánh để đọc các dòng cạnh nhau.
import subprocess
import sys as _sys

for TAG, D in (('B: asr_hard + LS=0', DEST_B), ('C: legacy + LS=0.1', DEST_C)):
    print(f'{"=" * 70}\n  {TAG}\n{"=" * 70}')
    subprocess.run([_sys.executable, '-m', 'src.evaluation.run_benchmark',
                    '--test_file', 'data/gold_test/gd1_basic/basic_test_multisource.jsonl',
                    '--model_name_or_path', str(D),
                    '--output_json', str(D / 'metrics.json'),
                    '--output_log', str(D / 'benchmark_log.txt'),
                    '--batch_size', '16',
                    '--num_beams', str(BEAMS),
                    '--repetition_penalty', str(REP),
                    '--length_penalty', str(LENP),
                    '--no_repeat_ngram_size', str(NGRAM)], check=True)

!python -m src.evaluation.compare_runs \
    --runs_root "{CKPT_ROOT}" \
    --output_md "{CKPT_ROOT / 'so_sanh_dot_train.md'}" \
    --sort_by exact_match
